In [24]:
import pandas as pd

# The raw data was pulled once on Oct 2, 2026 from the Wake County ArcGIS service
# and saved to data/raw/wake_restaurants.csv on GitHub. The code is kept here
# for the record but is not run, so the notebook only reads from GitHub.

# import requests
# url = "https://maps.wakegov.com/arcgis/rest/services/Inspections/RestaurantInspectionsOpenData/MapServer/0/query"
# rows, offset = [], 0
# while True:
#     params = {"where": "1=1", "outFields": "*", "returnGeometry": "false",
#               "resultOffset": offset, "resultRecordCount": 1000, "f": "json"}
#     r = requests.get(url, params=params, timeout=30).json()
#     batch = [f["attributes"] for f in r.get("features", [])]
#     if not batch:
#         break
#     rows += batch
#     offset += len(batch)
# pd.DataFrame(rows).to_csv("wake_restaurants.csv", index=False)

In [25]:
n_cities = clean["CITY"].nunique()
dictionary.loc[dictionary["Variable"] == "CITY", "Valid range or levels"] = (
    f"{n_cities} distinct values after cleaning (RTP is a business park label, not a town)"
)
display(dictionary)

,Variable,Type,Units,Valid range or levels,Missingness,Notes
0,HSISID,string,identifier,unique per facility,0 missing,"State facility ID, do not do math on it"
1,NAME,string,text,any text,0 missing,"Chains repeat, 203 repeated names"
2,CITY,category,label,17 distinct values after cleaning (RTP is a bu...,0 missing,"Mailing city, not always the legal city"
3,ZIP5,string,zip code,5 digit Wake County zips,0 missing,"Use this, not POSTALCODE, for the join"
4,FACILITYTYPE,category,label,10 levels from 1 Restaurant to 30 Meat Market,0 missing,"Only 2,729 of 4,182 rows are restaurants"
5,"X, Y",float,degrees longitude and latitude,Wake County area,1 row missing after cleaning,Missing means the address could not be matched
6,RESTAURANTOPENDATE,date,date,1991 to 2026,0 missing,Was milliseconds since 1970
7,GEOCODESTATUS,category,label,"M, T, U",0 missing,"99.9% matched, not the 85% I estimated in P1"
8,ADDRESS2,string,text,unit or suite,"3,543 missing (84.7%)",Missing means no unit number


In [26]:
print("Earliest open date:", clean["RESTAURANTOPENDATE"].min())
print("Latest open date:", clean["RESTAURANTOPENDATE"].max())
print("Dates after Oct 2, 2026:", (clean["RESTAURANTOPENDATE"] > "2026-10-02").sum())

Earliest open date: 1991-03-18 05:00:00
Latest open date: 2026-10-05 05:00:00
Dates after Oct 2, 2026: 1


## Checked for and not found

1. Sentinel values: one found, a coordinate of 0 on a single row (log row 4). No others in any column.
2. Total or subtotal rows: none. Every row is one facility.
3. Leading zeros lost: none. HSISID, PHONENUMBER and POSTALCODE were read as text.
4. Garbled text from the wrong encoding: none seen in names, addresses or cities.
5. Duplicate keys: none. OBJECTID and HSISID each have 4,182 distinct values for 4,182 rows.
6. Dates in more than one format: none. RESTAURANTOPENDATE is one numeric format.
7. Numbers stored as text: none, apart from identifiers that should be text.

In [27]:
DATA = "https://raw.githubusercontent.com/tbirru/dsa405-project/main/data/raw/"
df = pd.read_csv(DATA + "wake_restaurants.csv")
print(df.shape)
df.info()
print(df.isna().sum())
print((df.isna().mean() * 100).round(1))
print(df.nunique())
print("exact duplicates:", df.duplicated().sum())
df.head()

(4182, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4182 entries, 0 to 4181
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   OBJECTID            4182 non-null   int64  
 1   HSISID              4182 non-null   int64  
 2   NAME                4182 non-null   object 
 3   ADDRESS1            4182 non-null   object 
 4   ADDRESS2            639 non-null    object 
 5   CITY                4182 non-null   object 
 6   STATE               4182 non-null   object 
 7   POSTALCODE          4182 non-null   object 
 8   PHONENUMBER         3812 non-null   object 
 9   RESTAURANTOPENDATE  4182 non-null   int64  
 10  FACILITYTYPE        4182 non-null   object 
 11  X                   4182 non-null   float64
 12  Y                   4182 non-null   float64
 13  GEOCODESTATUS       4182 non-null   object 
 14  PERMITID            4182 non-null   object 
dtypes: float64(2), int64(3), object(10)
memory u

,OBJECTID,HSISID,NAME,ADDRESS1,ADDRESS2,CITY,STATE,POSTALCODE,PHONENUMBER,RESTAURANTOPENDATE,FACILITYTYPE,X,Y,GEOCODESTATUS,PERMITID
0,1,4092010037,AUBREY & PEEDIE'S GRILL,38 N MAIN ST,NaN,WENDELL,NC,27591,(919) 365-5528,672555600000,1 - Restaurant,-78.369456,35.782463,M,Not Provided
1,2,4092010018,ANGUS BARN,9401 GLENWOOD AVE,NaN,RALEIGH,NC,27628,(919) 781-2444,673246800000,1 - Restaurant,-78.764176,35.898364,M,Not Provided
2,3,4092010065,BISCUITVILLE,2426 WAKE FOREST RD,NaN,RALEIGH,NC,27608-1710,(919) 821-2060,672901200000,1 - Restaurant,-78.625690,35.815429,M,Not Provided
3,4,4092010070,BOJANGLES # 3,1013 NEW BERN AVE,NaN,RALEIGH,NC,27601-1605,(919) 821-3735,672555600000,1 - Restaurant,-78.622990,35.780248,M,Not Provided
4,5,4092010072,BOJANGLES #11,7525 KNIGHTDALE BLVD,NaN,KNIGHTDALE,NC,27545-9274,(919) 266-2768,673074000000,1 - Restaurant,-78.475672,35.799463,M,Not Provided


In [28]:
print(df["CITY"].value_counts(dropna=False))
print(df["FACILITYTYPE"].value_counts(dropna=False))
print(df["GEOCODESTATUS"].value_counts(dropna=False))
print(df["POSTALCODE"].astype(str).str.len().value_counts())
print(df["NAME"].duplicated().sum())
print(df.duplicated(subset=["NAME","ADDRESS1","CITY"]).sum())
print((df["X"]==0).sum(), (df["Y"]==0).sum())
print(df["RESTAURANTOPENDATE"].describe())

CITY
RALEIGH          2120
CARY              632
WAKE FOREST       265
APEX              247
MORRISVILLE       218
GARNER            143
FUQUAY-VARINA     139
HOLLY SPRINGS     128
KNIGHTDALE         96
WENDELL            63
ZEBULON            62
ROLESVILLE         40
FUQUAY VARINA      18
CLAYTON             3
WILLOW SPRING       2
ANGIER              2
NEW HILL            1
RTP                 1
FUQUAY              1
ROLESVILE           1
Name: count, dtype: int64
FACILITYTYPE
1 - Restaurant                     2729
2 - Food Stands                     757
3 - Mobile Food                     252
11 - Public School Lunch            195
30 - Meat Market                    105
16 - Institutional Food Service      43
4 - Push Carts                       42
14 - Limited Food                    30
5 - Private Schools Lunchroom        22
9 - Elderly Nutrition                 7
Name: count, dtype: int64
GEOCODESTATUS
M    4178
T       3
U       1
Name: count, dtype: int64
POSTALCODE
5     361

In [29]:
# Start: load the raw file, keeping IDs and zip codes as text
raw = pd.read_csv(DATA + "wake_restaurants.csv",
                  dtype={"POSTALCODE": str, "HSISID": str, "PHONENUMBER": str})
clean = raw.copy()   # work on a copy so the raw data stays untouched

# Step 1: zip codes. Some look like 27608 1710 (9 digits), some are 5 digits.
# Keep only the first 5 digits in a new column so they match the Census data.
zip_count = (clean["POSTALCODE"].str.len() == 10).sum()
clean["ZIP5"] = clean["POSTALCODE"].str[:5]

# Step 2: Fuquay Varina was spelled three ways. Make them all the same.
fuquay_rows = clean["CITY"].str.startswith("FUQUAY") & (clean["CITY"] != "FUQUAY VARINA")
fuquay_count = fuquay_rows.sum()
clean.loc[fuquay_rows, "CITY"] = "FUQUAY VARINA"

# Step 3: fix one typo, ROLESVILE should be ROLESVILLE
rolesville_rows = clean["CITY"] == "ROLESVILE"
rolesville_count = rolesville_rows.sum()
clean.loc[rolesville_rows, "CITY"] = "ROLESVILLE"

# Step 4: one restaurant has map coordinates of 0, which is a placeholder.
# Set them to missing instead of pretending they are real.
zero_rows = clean["X"] == 0
zero_count = zero_rows.sum()
clean.loc[zero_rows, ["X", "Y"]] = None

# Step 5: the open date is a huge number (milliseconds since 1970).
# Turn it into a real date.
clean["RESTAURANTOPENDATE"] = pd.to_datetime(clean["RESTAURANTOPENDATE"], unit="ms")

# Step 6: drop two columns that never change (every row is the same value)
clean = clean.drop(columns=["PERMITID", "STATE"])

# Check: did the row and column counts work out?
print("Counts:", zip_count, fuquay_count, rolesville_count, zero_count)
print("Rows:", len(raw), "before,", len(clean), "after")
print("Columns:", raw.shape[1], "before,", clean.shape[1], "after")

Counts: 565 140 1 1
Rows: 4182 before, 4182 after
Columns: 15 before, 14 after


In [30]:
print(df[["X", "Y"]].describe())
print(df["STATE"].value_counts(dropna=False))
print(df["PERMITID"].value_counts(dropna=False))
print(df["ADDRESS2"].notna().sum(), "rows have a unit number")

                 X            Y
count  4182.000000  4182.000000
mean    -78.660328    35.792453
std       1.222968     0.560216
min     -78.944982     0.000000
25%     -78.782861    35.752053
50%     -78.660441    35.796430
75%     -78.597864    35.855421
max       0.000000    36.049483
STATE
NC    4182
Name: count, dtype: int64
PERMITID
Not Provided    4182
Name: count, dtype: int64
639 rows have a unit number


In [31]:
log = pd.DataFrame([
 [1, "POSTALCODE", "Read as text, added ZIP5 with the first 5 characters", int(zip_count), "Nine digit zips will not match the 5 digit zips in the Census data, and my join needs one format", "Nothing, original column kept"],
 [2, "CITY", "Changed FUQUAY-VARINA (139 rows) and FUQUAY (1 row) to FUQUAY VARINA", int(fuquay_count), "One town was spelled three ways, which would split its counts", "The original spellings"],
 [3, "CITY", "Changed ROLESVILE to ROLESVILLE", int(rolesville_count), "A typo created a fake extra city", "Nothing"],
 [4, "X, Y", "Set coordinates of 0 to missing", int(zero_count), "0 is a placeholder for the one unmatched address, not a real location", "The placeholder value 0"],
 [5, "RESTAURANTOPENDATE", "Converted milliseconds since 1970 into dates", len(raw), "The raw numbers cannot be read or compared as dates", "Nothing"],
 [6, "HSISID, PHONENUMBER", "Read as text", len(raw), "They are identifiers, so arithmetic on them is meaningless", "Nothing"],
 [7, "PERMITID", "Dropped column", len(raw), "Every row says Not Provided, so it holds no information", "The label Not Provided"],
 [8, "STATE", "Dropped column", len(raw), "Every row is NC, so it never varies", "The constant NC"],
], columns=["#", "Column", "Change made", "Rows or cells affected", "Why", "What is lost"])
display(log)

print("Rows in raw file:", len(raw))
print("Removed as exact duplicates:", int(raw.duplicated().sum()))
print("Removed as near duplicates:", int(raw.duplicated(subset=["NAME", "ADDRESS1", "CITY"]).sum()))
print("Removed for other reasons: 0")
print("Rows in cleaned file:", len(clean))
print("Columns in raw file:", raw.shape[1], "| dropped: 2 | added: 1 | cleaned:", clean.shape[1])

assert len(clean) == len(raw)
assert clean.shape[1] == raw.shape[1] - 2 + 1

dictionary = pd.DataFrame([
 ["HSISID", "string", "identifier", "unique per facility", "0 missing", "State facility ID, do not do math on it"],
 ["NAME", "string", "text", "any text", "0 missing", "Chains repeat, 203 repeated names"],
 ["CITY", "category", "label", "19 towns after cleaning", "0 missing", "Mailing city, not always the legal city"],
 ["ZIP5", "string", "zip code", "5 digit Wake County zips", "0 missing", "Use this, not POSTALCODE, for the join"],
 ["FACILITYTYPE", "category", "label", "10 levels from 1 Restaurant to 30 Meat Market", "0 missing", "Only 2,729 of 4,182 rows are restaurants"],
 ["X, Y", "float", "degrees longitude and latitude", "Wake County area", "1 row missing after cleaning", "Missing means the address could not be matched"],
 ["RESTAURANTOPENDATE", "date", "date", "1991 to 2026", "0 missing", "Was milliseconds since 1970"],
 ["GEOCODESTATUS", "category", "label", "M, T, U", "0 missing", "99.9% matched, not the 85% I estimated in P1"],
 ["ADDRESS2", "string", "text", "unit or suite", "3,543 missing (84.7%)", "Missing means no unit number"],
], columns=["Variable", "Type", "Units", "Valid range or levels", "Missingness", "Notes"])
display(dictionary)

,#,Column,Change made,Rows or cells affected,Why,What is lost
0,1,POSTALCODE,"Read as text, added ZIP5 with the first 5 char...",565,Nine digit zips will not match the 5 digit zip...,"Nothing, original column kept"
1,2,CITY,Changed FUQUAY-VARINA (139 rows) and FUQUAY (1...,140,"One town was spelled three ways, which would s...",The original spellings
2,3,CITY,Changed ROLESVILE to ROLESVILLE,1,A typo created a fake extra city,Nothing
3,4,"X, Y",Set coordinates of 0 to missing,1,0 is a placeholder for the one unmatched addre...,The placeholder value 0
4,5,RESTAURANTOPENDATE,Converted milliseconds since 1970 into dates,4182,The raw numbers cannot be read or compared as ...,Nothing
5,6,"HSISID, PHONENUMBER",Read as text,4182,"They are identifiers, so arithmetic on them is...",Nothing
6,7,PERMITID,Dropped column,4182,"Every row says Not Provided, so it holds no in...",The label Not Provided
7,8,STATE,Dropped column,4182,"Every row is NC, so it never varies",The constant NC


Rows in raw file: 4182
Removed as exact duplicates: 0
Removed as near duplicates: 0
Removed for other reasons: 0
Rows in cleaned file: 4182
Columns in raw file: 15 | dropped: 2 | added: 1 | cleaned: 14


,Variable,Type,Units,Valid range or levels,Missingness,Notes
0,HSISID,string,identifier,unique per facility,0 missing,"State facility ID, do not do math on it"
1,NAME,string,text,any text,0 missing,"Chains repeat, 203 repeated names"
2,CITY,category,label,19 towns after cleaning,0 missing,"Mailing city, not always the legal city"
3,ZIP5,string,zip code,5 digit Wake County zips,0 missing,"Use this, not POSTALCODE, for the join"
4,FACILITYTYPE,category,label,10 levels from 1 Restaurant to 30 Meat Market,0 missing,"Only 2,729 of 4,182 rows are restaurants"
5,"X, Y",float,degrees longitude and latitude,Wake County area,1 row missing after cleaning,Missing means the address could not be matched
6,RESTAURANTOPENDATE,date,date,1991 to 2026,0 missing,Was milliseconds since 1970
7,GEOCODESTATUS,category,label,"M, T, U",0 missing,"99.9% matched, not the 85% I estimated in P1"
8,ADDRESS2,string,text,unit or suite,"3,543 missing (84.7%)",Missing means no unit number


## Provenance Brief

This dataset lists the food businesses that Wake County, North Carolina inspects, including restaurants, food stands, food trucks, school lunch kitchens and meat markets. The county government produces it as part of its public health work and publishes it as open data, updated daily, so residents can see which food businesses operate near them. I downloaded 4,182 records on October 2, 2026. The count differs slightly from the 4,102 I saw earlier because the county updates the list daily. Each record gives a business name, address, city, zip code, phone number, type, opening date and map location. It can show where food businesses are and how many operate in each area. It cannot show how clean or popular any business is, because it has no inspection scores or customer counts, and it covers only active businesses, so closed ones are missing.